# 06 – Market maturity: is the market still growing?

## Question
- **Question:** Is the Airbnb market in each city still growing, saturated or shrinking? When did today's listings enter, is there a COVID dip (2020–21) and a recovery, and where in the city is the market expanding?
- **Key caveat (survivorship):** the data is a snapshot of the listings that are online in June 2026. Listings that left the platform before that are invisible. So "entries per year" are always **entries of the listings that survived until 2026**, not the historical supply, and older years are under-counted. Every growth curve in this study is read with this in mind (section 4 shows how large the effect can be).
- **Predictions:** written by Claude, not by the team. The study plan contained four predictions written before anyone looked at the data; they are quoted in their sections. Where Claude had already seen numbers during a first exploration of the data (before this notebook was written), the section says so openly.
- **Variables:** `first_review` (proxy for the date a listing entered the market), `id` (listing number, used as a second clock), `hosts_time_as_host_years` / `hosts_time_as_host_months` (how long the host has been hosting; replaces `host_since`, which is empty), `minimum_nights` (rental model from study 04), `room_type` (as a category only), `license` (Barcelona), `neighbourhood_cleansed` / `neighbourhood_group_cleansed`, `latitude` / `longitude`.
- **Methods:** missing values and a check of the start-date proxy (the listing id as a clock); frequency table and bar chart of entry years and cohorts; entries indexed to 2019 = 100, with a survivorship reference band; non-linear regression (logistic S-curve vs exponential vs linear fit, R², AIC, residuals); chi-square tests of independence with Cramér's V (cohort × rental model, cohort × room type, cohort × licence in Barcelona); new hosts vs incumbent hosts; Mann-Whitney U test with rank-biserial r (distance of new listings to the market centre); Spearman correlations per neighbourhood; maps (choropleth of the share of new listings, categorical point map with clustering, animated heat map over the years, static small multiples); replication on pairs 2 and 3 with the same functions.

## 1. Load the data

Load the columns this study needs for pair 1 (Barcelona, Los Angeles), the discovery pair. `load_cities` adds a `city_name` column ordered by pair and parses the date columns, which are strings in the CSV. The same function loads all six cities in the replication. `CITY_COLORS` fixes one colour per city, as in study 05.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import NullFormatter, ScalarFormatter
from scipy import stats
from scipy.optimize import curve_fit

from src.cities import CITIES, cities_in_pair
from src.load import load_all, load_neighbourhoods
from src.maps import add_category_points, add_choropleth, add_heatmap, add_heatmap_over_time, base_map, save_map

FIG_DIR = Path("../figures")  # PNGs for the report
FIG_DIR.mkdir(exist_ok=True)

# Fixed colour per city (the colour follows the city in every chart)
CITY_COLORS = dict(zip([c.name for c in CITIES.values()],
                       ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))

cols = [
    "id", "host_id", "last_scraped", "first_review", "last_review", "number_of_reviews", "number_of_reviews_ltm",
    "host_since", "hosts_time_as_host_years", "hosts_time_as_host_months",
    "minimum_nights", "room_type", "license",
    "neighbourhood_cleansed", "neighbourhood_group_cleansed", "latitude", "longitude",
]
DATE_COLS = ["last_scraped", "first_review", "last_review", "host_since"]


def load_cities(keys: list[str]) -> pd.DataFrame:
    """Load the listings of the given cities, with a `city_name` column ordered like `keys` (by pair)
    and the date columns parsed (they are strings in the CSV)."""
    data = load_all(keys, usecols=cols)
    data["city_name"] = pd.Categorical(data["city"].map({k: c.name for k, c in CITIES.items()}),
                                       categories=[CITIES[k].name for k in keys], ordered=True)
    for col in DATE_COLS:
        data[col] = pd.to_datetime(data[col])
    return data


# Discovery on pair 1 first; the replication at the end uses the same functions for all six cities
df = load_cities(cities_in_pair(1))
df.groupby("city_name").size()

## 2. Data check: when did a listing enter the market?

The data has no "listing created" date. Two columns come close:
- `first_review`, the date of the listing's first review. A listing gets its first review after its first stay, so this is the date the listing **entered the market** (with a lag of weeks or months after it was created). It is missing for listings that have never been reviewed.
- `host_since`, the date the host joined. This is host-level: a host who joined in 2015 can have created this listing in 2025.

The table shows the share of missing values per column.

**Result:** `first_review` is missing for **23.2% (Barcelona) and 25.7% (Los Angeles)** of listings, exactly the listings without any review (`last_review` is missing for the same rows). **`host_since` is empty in every city (100% missing):** in the 2026 Inside Airbnb schema it has been replaced by `hosts_time_as_host_years` and `hosts_time_as_host_months`, the time the host has been hosting at the moment of the scrape (missing for only 0.5–0.6%). The study therefore derives the host's start year from these two columns and uses `first_review` as the listing's entry date.

In [ ]:
# Share of missing values (%) per date column and city
df[["first_review", "last_review", "host_since", "hosts_time_as_host_years"]].isna().groupby(df["city_name"]).mean().mul(100).round(1).T

### 2b. The study's columns

The cell adds all derived columns in one function, so the replication can call the same function:
- `entry_year` = year of the first review, and `cohort`, five entry periods: before 2015, 2015–2019, **2020–2021 (COVID)**, 2022–2023 and 2024–2026. 2026 covers only January to June.
- `rental_model` = the three groups of study 04 (short stay 1–6 nights, weekly 7–29, monthly+ 30+), from `minimum_nights`.
- `host_start_year` = scrape date minus the host's time as a host.
- `is_new` = the first review was at most 24 months before the scrape.
- `id_year` and `dist_km`, explained in sections 2c and 8.

Cohort colours: one orange hue from light (oldest) to dark (newest), because the cohorts are ordered. Blue stays reserved for magnitudes (e.g. the share of new listings on the maps), so the two never mix on one map.

In [ ]:
COHORT_LABELS = ["before 2015", "2015-2019", "2020-2021 (COVID)", "2022-2023", "2024-2026"]
COHORT_BINS = [0, 2014, 2019, 2021, 2023, 2026]  # (0, 2014] = before 2015, (2014, 2019] = 2015-2019, ...
# One orange hue from light (oldest) to dark (newest); blue stays reserved for magnitudes such as shares
COHORT_COLORS = dict(zip(COHORT_LABELS, ["#f29a6e", "#eb6834", "#c9521f", "#9c3c13", "#6b270a"]))
RENTAL_LABELS = ["Short stay (1-6 nights)", "Weekly (7-29)", "Monthly+ (30+)"]  # the groups of study 04
NEW_MONTHS = 24  # a listing is "new" if its first review is at most 24 months before the scrape


def id_implied_year(data: pd.DataFrame, n_bins: int = 50) -> pd.Series:
    """Entry year implied by the listing id, for every listing (reviewed or not).

    Per city, the reviewed listings are cut into `n_bins` groups of equal size by id; each group gets the
    median entry year of its listings. Every listing then gets the value of the id group it falls into.
    """
    implied = pd.Series(np.nan, index=data.index)
    for _, g in data.groupby("city", sort=False):
        reviewed = g[g["first_review"].notna()]
        groups = pd.qcut(reviewed["id"], n_bins, duplicates="drop")
        median_year = reviewed["first_review"].dt.year.groupby(groups, observed=True).median()
        upper_edges = np.array([interval.right for interval in median_year.index])
        position = np.clip(np.searchsorted(upper_edges, g["id"]), 0, len(upper_edges) - 1)
        implied.loc[g.index] = median_year.to_numpy()[position]
    return implied


def distance_to_centre(data: pd.DataFrame) -> pd.Series:
    """Distance in km from each listing to its city's market centre (median latitude and longitude of all
    listings), by the haversine formula for distances on a sphere."""
    centre = data.groupby("city")[["latitude", "longitude"]].transform("median")
    lat1, lon1, lat2, lon2 = (np.radians(s) for s in
                              [data["latitude"], data["longitude"], centre["latitude"], centre["longitude"]])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


def add_entry_columns(data: pd.DataFrame) -> pd.DataFrame:
    """Add the columns of this study: entry year and cohort, rental model, host start year, "new" flag,
    id-implied entry year and distance to the market centre."""
    data = data.copy()
    data["entry_year"] = data["first_review"].dt.year  # NaN for listings without a review
    data["cohort"] = pd.cut(data["entry_year"], bins=COHORT_BINS, labels=COHORT_LABELS)
    data["rental_model"] = pd.cut(data["minimum_nights"], bins=[0, 6, 29, np.inf], labels=RENTAL_LABELS)
    data["is_monthly"] = data["minimum_nights"] >= 30
    # Time as a host at the scrape, in months -> the year the host started hosting
    host_months = data["hosts_time_as_host_years"] * 12 + data["hosts_time_as_host_months"]
    data["host_start_year"] = (data["last_scraped"] - pd.to_timedelta(host_months * 30.44, unit="D")).dt.year
    # False for listings without a review; analyses of "new" listings use reviewed listings only
    data["is_new"] = (data["last_scraped"] - data["first_review"]).dt.days <= NEW_MONTHS * 30.44
    data["id_year"] = id_implied_year(data)
    data["dist_km"] = distance_to_centre(data)
    return data


df = add_entry_columns(df)
df[["city_name", "id", "first_review", "entry_year", "cohort", "id_year", "host_start_year", "is_new", "dist_km"]].head()

### 2c. Are the listings without a review new? The listing id as a clock

About a quarter of the listings have no first review, so they have no entry year. If they are mostly new listings that have not had a guest yet, dropping them under-counts the most recent years. If they are mostly old, inactive listings, it does not.

Airbnb gives listings increasing numbers: small sequential ids until about 2021, then long 18-digit ids. If the id grows with time, it is a second clock that also works for listings without a review. The function `id_implied_year` cuts each city's reviewed listings into 50 equal groups by id and gives each group the median entry year of its listings; every listing, reviewed or not, then gets the year of its id group. The table checks how well this works on the reviewed listings (Spearman ρ between id and first-review date, and how often the implied year is within one year of the true entry year), then describes the listings without a review.

**Prediction:** no blind prediction here. During a first look at the data Claude had already seen that the id and the first-review date are strongly correlated and that most listings without a review have recent ids.

**Result:**
- **The id is a good clock.** Spearman ρ between id and first-review date = **0.955 (Barcelona) and 0.964 (Los Angeles)**. For reviewed listings, the id-implied year is within one year of the true entry year in **91.6% and 94.6%** of cases. The chart shows two regimes: sequential ids up to about 10⁸ until early 2022, then 18-digit ids. A few listings have a new id but an old first review (dots in the upper left), probably listings that were re-created and kept their reviews.
- **Listings without a review are mostly new.** **57.2% (Barcelona) and 60.2% (Los Angeles)** of them have an id-implied entry year of 2025–2026, against 25.0% and 31.1% of the reviewed listings. Only 15.5% and 10.6% look older than 2022.
- **They are also mostly monthly+:** 61.3% and 70.3%, against 29.8% and 38.7% of reviewed listings. A 30+ listing hosts few stays, so it waits longer for its first review.
- **Consequence for the study:** counting reviewed listings only under-counts the most recent entries (2025–2026) and under-counts monthly+ listings among them. The direction of this bias is known, so each later section says how it affects the result. Section 6 repeats the key chart with the id-implied year for all listings.

In [ ]:
def unreviewed_profile(data: pd.DataFrame) -> pd.DataFrame:
    """Per city: how well the id predicts the entry year, and what the listings without a review look like."""
    rows = {}
    for city, g in data.groupby("city_name"):
        rev, unrev = g[g["first_review"].notna()], g[g["first_review"].isna()]
        rho, _ = stats.spearmanr(rev["id"], rev["first_review"].astype("int64"))
        rows[city] = {
            "unreviewed_pct": len(unrev) / len(g) * 100,
            "rho_id_vs_first_review": rho,
            "id_year_within_1y_pct": ((rev["id_year"] - rev["entry_year"]).abs() <= 1).mean() * 100,
            "reviewed_entered_2025_26_pct": (rev["entry_year"] >= 2025).mean() * 100,
            "unreviewed_implied_2025_26_pct": (unrev["id_year"] >= 2025).mean() * 100,
            "unreviewed_implied_before_2022_pct": (unrev["id_year"] < 2022).mean() * 100,
            "reviewed_monthly_pct": rev["is_monthly"].mean() * 100,
            "unreviewed_monthly_pct": unrev["is_monthly"].mean() * 100,
        }
    return pd.DataFrame(rows).T.round(3)


unreviewed_profile(df)

In [ ]:
def plot_id_clock(data: pd.DataFrame, filename: str, sample: int = 4000) -> None:
    """Listing id (log scale) against the date of the first review, one panel per city (random sample)."""
    city_names = data["city_name"].unique()
    fig, axes = plt.subplots(1, len(city_names), figsize=(4.6 * len(city_names) + 0.5, 3.6), sharey=True, squeeze=False)
    for ax, city in zip(axes.flat, city_names):
        rev = data[(data["city_name"] == city) & data["first_review"].notna()]
        s = rev.sample(min(sample, len(rev)), random_state=0)
        ax.scatter(s["first_review"], s["id"], s=3, alpha=0.35, color=CITY_COLORS[city], linewidth=0, rasterized=True)
        ax.set_yscale("log")
        ax.set_title(city)
        ax.set_xlabel("Date of the first review")
        ax.grid(alpha=0.3)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    axes.flat[0].set_ylabel("Listing id (log scale)")
    fig.suptitle(f"The listing id grows with time (random sample of {sample:,} reviewed listings per city)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_id_clock(df, "study06_id_clock.png")

## 3. When did today's listings enter?

A frequency table of the entry cohorts (share of the city's reviewed listings), the number of entries per year around COVID, and a bar chart of entries per year (one panel per city, as a share of the city's reviewed listings, so the two cities share one scale). The 2026 bar is lighter: it covers January to June only.

Remember what the bars measure: **listings that entered in year X and are still online in June 2026.** A bar is low either because few listings entered that year or because many of them have left since.

**Prediction (from the study plan, written by Claude before looking at the data):** a COVID dip in 2020 in every city. (The yearly counts were then seen during the first exploration, before this notebook was written.)

**Result:**
- **Young stock:** **37.3% (Barcelona) and 45.1% (Los Angeles)** of today's reviewed listings entered in 2024–2026, and only 6.7% and 3.2% before 2015. The median surviving listing entered in 2022 (Barcelona) and 2023 (Los Angeles).
- **COVID dip, confirmed:** entries fell by **49.0% (Barcelona) and 29.8% (Los Angeles)** from 2019 to 2020. Both cities were back above the 2019 level already in 2021 (+2.6% and +13.1%). Since then, entries have risen every year, and 2025 is the year with the most entries in both cities.
- 2026 (January to June) already has 76% (Barcelona) and 61% (Los Angeles) of the 2025 entries, but the first-review lag cuts it short; it is left out of the trends below.

**Interpretation (draft for the team to check):** the dip is in the date of the **first review**, not of the creation. A listing created in late 2019 that got its first guest only in 2021 counts as a 2021 entry. So the 2020 dip mixes fewer new listings with fewer first stays. The quick recovery in 2021 fits both: domestic travel returned, and listings waiting for their first guest got it.

In [ ]:
def entries_per_year(data: pd.DataFrame) -> pd.DataFrame:
    """Number of listings per entry year (rows) and city (columns), reviewed listings only."""
    dated = data.dropna(subset=["entry_year"])
    return pd.crosstab(dated["entry_year"].astype(int), dated["city_name"])


def cohort_table(data: pd.DataFrame) -> pd.DataFrame:
    """Share (%) of each entry cohort among the city's reviewed listings."""
    dated = data.dropna(subset=["cohort"])
    t = pd.crosstab(dated["city_name"], dated["cohort"], normalize="index").mul(100).round(1)
    t.insert(0, "reviewed_listings", dated.groupby("city_name").size())
    return t


def covid_dip(counts: pd.DataFrame) -> pd.DataFrame:
    """Entries in 2019-2021 per city and their change against 2019 (%), and the year with most entries."""
    return pd.DataFrame({
        "entries_2019": counts.loc[2019],
        "entries_2020": counts.loc[2020],
        "change_2020_vs_2019_pct": (counts.loc[2020] / counts.loc[2019] - 1) * 100,
        "entries_2021": counts.loc[2021],
        "change_2021_vs_2019_pct": (counts.loc[2021] / counts.loc[2019] - 1) * 100,
        "peak_year_to_2025": counts.loc[:2025].idxmax(),
    }).round(1)


counts = entries_per_year(df)
display(cohort_table(df))
display(covid_dip(counts))
counts.loc[2010:].T

In [ ]:
def plot_entries(counts: pd.DataFrame, filename: str, first: int = 2010) -> None:
    """Entries per year as % of the city's reviewed listings, one panel per city. 2026 is a partial year."""
    city_names = list(counts.columns)
    ncols = min(3, len(city_names))
    nrows = -(-len(city_names) // ncols)  # ceiling division
    share = counts.loc[first:] / counts.sum() * 100
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols + 0.5, 3.4 * nrows), sharey=True, squeeze=False)
    for ax, city in zip(axes.flat, city_names):
        bars = ax.bar(share.index, share[city], width=0.8, color=CITY_COLORS[city])
        bars[-1].set_alpha(0.4)  # 2026: January to June only
        ax.text(share.index[-1], share[city].iloc[-1], "Jan-\nJun", ha="center", va="bottom", fontsize=7)
        ax.axvspan(2019.5, 2021.5, color="grey", alpha=0.12, linewidth=0)
        ax.text(2020.5, 0.97, "COVID", transform=ax.get_xaxis_transform(), ha="center", va="top", fontsize=8,
                color="#52514e")
        ax.set_xticks(range(first, share.index[-1] + 1, 2))
        ax.set_title(city)
        ax.set_xlabel("Year of the first review")
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("% of today's reviewed listings")
    for ax in axes.flat[len(city_names):]:
        ax.set_visible(False)
    fig.suptitle("When did today's listings enter? (survivors only)")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_entries(counts, "study06_entries_per_year.png")

## 4. Growth over time, indexed to 2019 = 100, and the survivorship trap

Barcelona and Los Angeles differ in size, so the entries per year are divided by the city's entries in 2019 (the last year before COVID) and multiplied by 100. An index of 200 in 2025 means twice as many surviving entries as in 2019. The y-axis is logarithmic, so equal ratios look equally steep in both cities.

**Survivorship reference.** Suppose the market did not grow at all: the same number of listings entered every year, and every year a share *d* of all listings left the platform. A listing from year *t* would then still be online in 2026 with probability (1 − *d*)^(2026 − *t*). Today's snapshot would show a **rising** index although the market is flat. The grey band shows this "constant market" for *d* = 10% and 20% per year. These exit rates are **assumptions** for illustration; the data cannot measure them (to verify, e.g. with an older Inside Airbnb snapshot). A city line inside the band is compatible with a flat market; only a line steeper than the band shows real growth, and a falling line shows fewer entries even before any exits.

The same logic gives a simple check of recent momentum: in a constant market, 2025 would have (1 − *d*)^−2 times more survivors than 2023, i.e. 1.23 (*d* = 10%) to 1.56 (*d* = 20%). The `momentum` table compares the observed ratio with that range.

**Prediction (from the study plan, written by Claude before looking at the data):** a flatter curve in Barcelona than in Los Angeles, because Barcelona has issued no new tourist licences since 2014 (study 04).

**Result:**
- **Index 2025 (2019 = 100): Barcelona 216, Los Angeles 362.** Barcelona's curve is flatter, as predicted. Before 2019 the two cities are similar (2015: 62 and 52).
- **But both lines lie inside the grey band.** In 2025 the band spans 188 (10% exits) to 381 (20% exits). The index alone therefore cannot tell growth from churn: a flat market with 10–20% yearly exits would produce the same picture.
- **Recent momentum:** entries 2025 / 2023 = **1.34 in Barcelona**, inside the constant-market range (up to 1.56), so it is *compatible with a flat market*. **Los Angeles: 1.67**, above the range, so it is *growing* even under the 20% exit assumption.

**Interpretation (draft for the team to check):** the prediction holds, but more weakly than it sounds. Barcelona's survivor curve is what a market with no net growth and normal churn would show. Los Angeles shows real growth only in the last two years. The 10–20% exit rates are assumptions; if the true churn is lower, both cities are growing; if it is higher, both could be flat. An older Inside Airbnb snapshot would settle this (to verify).

In [ ]:
BASE_YEAR = 2019
EXIT_RATES = (0.10, 0.20)  # assumed yearly exit rates for the "constant market" reference (not measured)


def entry_index(counts: pd.DataFrame, base: int = BASE_YEAR) -> pd.DataFrame:
    """Entries per year indexed to the base year (= 100)."""
    return counts.div(counts.loc[base]).mul(100)


def momentum(counts: pd.DataFrame) -> pd.DataFrame:
    """Entries in 2025 against 2023, compared with what a constant market with listing exits would show.

    If the same number of listings entered every year and a share d of listings left every year, today's
    snapshot would hold (1 - d)^-2 times more survivors from 2025 than from 2023 (1.23 for d = 10%,
    1.56 for d = 20%). A ratio below 1 means fewer entries even before any exits are taken into account.
    """
    ratio = counts.loc[2025] / counts.loc[2023]
    flat_max = (1 - EXIT_RATES[1]) ** -2
    reading = np.select([ratio < 1, ratio <= flat_max], ["shrinking", "compatible with a flat market"], "growing")
    return pd.DataFrame({"entries_2023": counts.loc[2023], "entries_2025": counts.loc[2025],
                         "ratio_2025_vs_2023": ratio.round(2), "reading": reading})


display(entry_index(counts).loc[[2015, 2017, 2019, 2020, 2021, 2023, 2025]].round(0).T)
momentum(counts)

In [ ]:
def plot_index(counts: pd.DataFrame, filename: str, first: int = 2012, last: int = 2025) -> None:
    """Entries per year indexed to 2019 = 100 (log scale), one line per city, with the band a constant
    market would show under the assumed exit rates."""
    idx = entry_index(counts).loc[first:last]
    years = idx.index.to_numpy()
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    band = [100 * (1 - d) ** (BASE_YEAR - years) for d in EXIT_RATES]
    ax.fill_between(years, band[0], band[1], color="grey", alpha=0.2, linewidth=0,
                    label=f"Constant market, {EXIT_RATES[0]:.0%}-{EXIT_RATES[1]:.0%} of listings leave per year (assumed)")
    for city in idx.columns:
        ax.plot(years, idx[city], color=CITY_COLORS[city], linewidth=2, marker="o", markersize=4, label=city)
    ax.axhline(100, color="grey", linewidth=0.8)
    ax.axvspan(2019.5, 2021.5, color="grey", alpha=0.08, linewidth=0)
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_yticks([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000])
    ax.set_xlabel("Year of the first review (2026 left out: partial year)")
    ax.set_ylabel(f"Entries of today's listings ({BASE_YEAR} = 100, log scale)")
    ax.set_title(f"Entries per year, indexed to {BASE_YEAR} = 100")
    ax.legend(frameon=False, fontsize=8, loc="upper left")
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_index(counts, "study06_entries_index.png")

## 5. Non-linear regression: an S-curve for the market?

A market that is saturating follows an **S-curve** (logistic growth): slow start, fast growth, then a slowdown towards a ceiling. The logistic model is

*N*(*t*) = *K* / (1 + e^(−*r*(*t* − *t*₀)))

with *K* = saturation level, *r* = growth rate and *t*₀ = **inflection year**, the year with the fastest growth (after *t*₀ growth slows down). It is fitted with `scipy.optimize.curve_fit` (non-linear least squares) to the **cumulative** number of today's listings by entry year, 2010–2025 (2026 is partial). For comparison, an exponential model (growth without a ceiling) and a linear model (constant growth) are fitted to the same points. R² measures the fit; because the logistic model has one parameter more, the models are also compared by **AIC** = *n* ln(RSS/*n*) + 2*k* (lower is better; it penalises extra parameters).

How to read it with survivorship: survivorship adds listings to the *recent* years, so it bends the curve **upwards** and makes it look exponential even in a flat market. An inflection year in the past is therefore the strong signal: it means entries were already slowing down **despite** survivorship. An inflection year in the future (with a huge, uncertain *K*) means the logistic fit has collapsed into an exponential one: no slowdown is visible yet, but that may be partly survivorship.

**Prediction:** no blind prediction for the fitted numbers (Claude saw them during the first exploration). The study-plan prediction from section 4 applies: a flatter curve in Barcelona than in Los Angeles.

**Result:**
- **The linear model is clearly wrong** (R² 0.925 and 0.858; its residuals form a U shape, a systematic error, not noise). The logistic and exponential models both fit well (R² 0.985–0.995, residuals within ±5% of the final count, with a bump before 2020 and a dip in 2020–21 from COVID). AIC prefers the logistic model in both cities.
- **Barcelona:** inflection year **t₀ = 2024.6 ± 1.5**, saturation level *K* = 19,300 ± 4,300 listings, 1.8 times today's count of reviewed listings.
- **Los Angeles:** t₀ = **2028.8 ± 2.8**, in the future, with *K* = 110,000 ± 58,000 (3.8 times today's count). The logistic fit is practically an exponential one (R² 0.995 vs 0.994): no slowdown is visible yet.

**Interpretation (draft for the team to check):** Barcelona's survivor curve starts to bend around now, Los Angeles's does not, which matches section 4. Barcelona's inflection is close to the present and has a standard error of 1.5 years, so "the slowdown has started" is a cautious reading, not a proof: yearly entries are still rising (+15% in 2025). *K* should not be read as a forecast of the maximum number of listings. It is the ceiling of the survivors' curve, and its error is large.

In [ ]:
FIT_YEARS = (2010, 2025)  # complete years only


def logistic(t, K, r, t0):
    """S-curve: K = saturation level, r = growth rate, t0 = inflection year (fastest growth)."""
    return K / (1 + np.exp(-r * (t - t0)))


def exponential(t, a, b):
    return a * np.exp(b * (t - FIT_YEARS[0]))


def fit_city(cum: pd.Series) -> dict:
    """Fit the logistic, exponential and linear model to one city's cumulative entries."""
    t, y = cum.index.to_numpy(dtype=float), cum.to_numpy(dtype=float)
    # K must be at least today's count; t0 is free between 2000 and 2100
    p_log, cov_log = curve_fit(logistic, t, y, p0=[2 * y[-1], 0.3, 2022],
                               bounds=([y[-1], 0.01, 2000], [100 * y[-1], 3, 2100]), maxfev=20000)
    p_exp, _ = curve_fit(exponential, t, y, p0=[max(y[0], 1), 0.3], maxfev=20000)
    p_lin = np.polyfit(t, y, 1)
    preds = {"logistic": logistic(t, *p_log), "exponential": exponential(t, *p_exp), "linear": np.polyval(p_lin, t)}
    n_params = {"logistic": 3, "exponential": 2, "linear": 2}
    out = {"t": t, "y": y, "preds": preds, "p_log": p_log, "se_log": np.sqrt(np.diag(cov_log))}
    for model, pred in preds.items():
        rss = ((y - pred) ** 2).sum()
        out[f"r2_{model}"] = 1 - rss / ((y - y.mean()) ** 2).sum()
        out[f"aic_{model}"] = len(y) * np.log(rss / len(y)) + 2 * n_params[model]  # lower = better
    return out


def growth_models(counts: pd.DataFrame) -> pd.DataFrame:
    """Model comparison and logistic parameters per city (cumulative entries of today's listings)."""
    rows = {}
    for city in counts.columns:
        f = fit_city(counts.loc[FIT_YEARS[0]:FIT_YEARS[1], city].cumsum())
        aic = {m: f[f"aic_{m}"] for m in ["logistic", "exponential", "linear"]}
        K, r, t0 = f["p_log"]
        rows[city] = {
            "r2_logistic": f["r2_logistic"], "r2_exponential": f["r2_exponential"], "r2_linear": f["r2_linear"],
            "best_by_aic": min(aic, key=aic.get),
            "inflection_year_t0": t0, "t0_se": f["se_log"][2],
            "saturation_K": K, "K_se": f["se_log"][0], "K_vs_count_2025": K / f["y"][-1], "rate_r": r,
        }
    return pd.DataFrame.from_dict(rows, orient="index").round(
        {"r2_logistic": 3, "r2_exponential": 3, "r2_linear": 3, "inflection_year_t0": 1, "t0_se": 1,
         "saturation_K": 0, "K_se": 0, "K_vs_count_2025": 1, "rate_r": 2})


growth_models(counts)

In [ ]:
MODEL_STYLES = {"logistic": "-", "exponential": "--", "linear": ":"}  # line style = model (all dark grey)


def plot_growth_fits(counts: pd.DataFrame, filename: str) -> None:
    """Cumulative entries (points) with the three fitted models (lines), and the residuals below,
    one panel per city."""
    city_names = list(counts.columns)
    ncols = min(3, len(city_names))
    nrows = -(-len(city_names) // ncols)
    fig = plt.figure(figsize=(5 * ncols + 0.5, 5.6 * nrows))
    outer = fig.add_gridspec(nrows, ncols, hspace=0.35, wspace=0.3)
    for i, city in enumerate(city_names):
        inner = outer[divmod(i, ncols)].subgridspec(2, 1, height_ratios=[3, 1.2], hspace=0.08)
        ax = fig.add_subplot(inner[0])
        ax_res = fig.add_subplot(inner[1], sharex=ax)
        f = fit_city(counts.loc[FIT_YEARS[0]:FIT_YEARS[1], city].cumsum())
        ax.scatter(f["t"], f["y"] / 1000, s=28, color=CITY_COLORS[city], zorder=3, label="Observed (survivors)",
                   edgecolor="white", linewidth=0.8)
        for model, pred in f["preds"].items():
            ax.plot(f["t"], pred / 1000, color="#333333", linestyle=MODEL_STYLES[model], linewidth=1.5,
                    label=f"{model} (R² {f[f'r2_{model}']:.3f})")
            ax_res.plot(f["t"], (f["y"] - pred) / f["y"][-1] * 100, color="#333333",
                        linestyle=MODEL_STYLES[model], linewidth=1.5)
        ax_res.axhline(0, color="grey", linewidth=0.8)
        ax.set_title(f"{city} (logistic inflection: {f['p_log'][2]:.1f})")
        ax.set_ylabel("Cumulative listings (thousands)")
        ax_res.set_ylabel("Residual (%)")
        ax_res.set_xlabel("Year of the first review")
        ax_res.set_xticks(range(FIT_YEARS[0], FIT_YEARS[1] + 1, 3))
        ax.tick_params(labelbottom=False)
        ax.legend(frameon=False, fontsize=8, loc="upper left")
        for a in (ax, ax_res):
            a.grid(alpha=0.3)
            a.set_axisbelow(True)
            a.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Cumulative entries of today's listings: logistic vs exponential vs linear fit", y=0.93)
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_growth_fits(counts, "study06_growth_fits.png")

## 6. Who enters? Rental model and room type by cohort

Do the listings that entered recently differ from the old ones? Two **chi-square tests of independence** per city: cohort × monthly+ (yes/no, the 30+ group of study 04) and cohort × entire home (yes/no). The test asks whether the share of monthly+ listings (or of entire homes) is the same in every cohort. With thousands of listings even tiny differences are significant, so **Cramér's V** is the effect size (0 = no association, 1 = perfect; for a table with two columns roughly 0.1 small, 0.3 medium, 0.5 large). Both variables are binary on purpose: the chi-square approximation needs large enough expected counts, and the weekly group, hotel rooms and shared rooms are too rare for that in small cohorts. The column `valid` applies **Cochran's rule** (no expected count below 1, at most 20% of the cells below 5).

**Caveat on the timing:** `minimum_nights` is today's value, not the value when the listing entered. An old listing that switched to 30+ after a rule change counts as monthly+. The test therefore describes what each cohort offers **today**.

**Caveat on the proxy:** a monthly+ listing hosts at most 12 stays a year, so it gets its first review later, or never. Monthly+ listings are therefore missing more often from the reviewed listings, especially in the recent years. The chart shows the monthly+ share per entry year twice: by the first-review year (reviewed listings only) and by the id-implied year (all listings, from section 2c). The gap between the two lines shows how large this bias is.

**Prediction (written before running, by Claude):** in Los Angeles, the monthly+ share is higher among listings that entered after 2019, when the Home-Sharing Ordinance came into force (study 04). Room type: a small effect (V < 0.1) in both cities. (Barcelona's monthly+ share per entry year was already seen during the first exploration, so there is no blind prediction for Barcelona.)

**Result:**
- **The two cities move in opposite directions.**
  - **Barcelona:** the monthly+ share rises from **7.7% (before 2015) to 38.0% (2024–2026)**, Cramér's V = 0.220.
  - **Los Angeles:** it **falls** from **62.8% to 27.2%**, V = 0.252 (small to medium effect in both cities, all tests valid by Cochran's rule).
  - **The Los Angeles prediction was wrong in direction.**
- **Room type:** small differences. Entire homes: Barcelona 93.7% before 2015, then 71–79% (V = 0.134); Los Angeles 70–81% (V = 0.085). The prediction V < 0.1 holds for Los Angeles only.
- **Proxy check (chart):** the id-implied line lies above the first-review line in recent years: by about 10–13 points in Barcelona (2022–2024) and about 15 points in Los Angeles (2024–2026). So the first review indeed under-counts recent monthly+ listings. **The direction of both trends stays the same**: Los Angeles still falls (to about 38–42%), Barcelona still rises until 2023.

**Interpretation (draft for the team to check):** because `minimum_nights` is today's value, a cohort's monthly+ share mixes **what entered** with **what survived and how it changed since**. For Barcelona, the old cohorts are short-stay listings with a tourist licence (section 6b), a valuable asset that keeps them on the platform. For Los Angeles, see section 6c.

In [ ]:
def cohort_profile(data: pd.DataFrame) -> pd.DataFrame:
    """Per city and cohort: number of listings, share of monthly+ listings and of entire homes (%)."""
    dated = data.dropna(subset=["cohort"])
    g = dated.groupby(["city_name", "cohort"])
    return pd.DataFrame({
        "listings": g.size(),
        "monthly_pct": g["is_monthly"].mean() * 100,
        "entire_home_pct": g["room_type"].agg(lambda s: (s == "Entire home/apt").mean() * 100),
    }).round(1).unstack("city_name")


def cramers_v(table: pd.DataFrame) -> dict:
    """Chi-square test of independence on a contingency table, with Cramer's V as the effect size.

    `valid` applies Cochran's rule: no expected count below 1 and at most 20% of the cells below 5;
    otherwise the chi-square approximation of the p-value is not reliable.
    """
    chi2, p, dof, expected = stats.chi2_contingency(table)
    v = np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))
    below_5 = (expected < 5).mean() * 100
    return {"n": int(table.to_numpy().sum()), "chi2": chi2, "dof": dof, "p_value": p, "cramers_v": v,
            "min_expected": expected.min(), "cells_below_5_pct": below_5,
            "valid": bool(expected.min() >= 1 and below_5 <= 20)}


def cohort_tests(data: pd.DataFrame) -> pd.DataFrame:
    """Chi-square tests per city: cohort x monthly+ (yes/no) and cohort x entire home (yes/no).
    Binary variables keep the expected counts large enough in small cohorts."""
    dated = data.dropna(subset=["cohort", "minimum_nights"])
    entire_home = dated["room_type"] == "Entire home/apt"
    rows = {}
    for city, idx in dated.groupby("city_name").groups.items():
        rows[(city, "cohort x monthly+")] = cramers_v(pd.crosstab(dated.loc[idx, "cohort"], dated.loc[idx, "is_monthly"]))
        rows[(city, "cohort x entire home")] = cramers_v(pd.crosstab(dated.loc[idx, "cohort"], entire_home.loc[idx]))
    return pd.DataFrame.from_dict(rows, orient="index").round(3)


display(cohort_profile(df))
cohort_tests(df)

In [ ]:
def plot_monthly_by_year(data: pd.DataFrame, filename: str, first: int = 2014) -> None:
    """Share of monthly+ listings per entry year, one panel per city. Solid: entry year from the first review
    (reviewed listings only). Dashed: entry year implied by the id (all listings, also those never reviewed)."""
    city_names = data["city_name"].unique()
    ncols = min(3, len(city_names))
    nrows = -(-len(city_names) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols + 0.5, 3.4 * nrows), sharey=True, squeeze=False)
    for ax, city in zip(axes.flat, city_names):
        g = data[data["city_name"] == city]
        by_review = g.dropna(subset=["entry_year"]).groupby("entry_year")["is_monthly"].mean().mul(100).loc[first:]
        by_id = g.dropna(subset=["id_year"]).groupby("id_year")["is_monthly"].mean().mul(100).loc[first:]
        ax.plot(by_review.index, by_review, color=CITY_COLORS[city], linewidth=2, marker="o", markersize=4,
                label="By first review (reviewed listings)")
        ax.plot(by_id.index, by_id, color=CITY_COLORS[city], linewidth=1.5, linestyle="--",
                label="By id-implied year (all listings)")
        ax.set_ylim(0, 100)
        ax.set_title(city)
        ax.set_xlabel("Entry year (2026: partial)")
        ax.grid(alpha=0.3)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("Monthly+ listings (% of entry year)")
    for ax in axes.flat[len(city_names):]:
        ax.set_visible(False)
    # one legend for all panels, in neutral grey (the colour only identifies the city)
    handles = [plt.Line2D([], [], color="#52514e", linewidth=2, marker="o", markersize=4),
               plt.Line2D([], [], color="#52514e", linewidth=1.5, linestyle="--")]
    fig.legend(handles, ["Entry year from the first review (reviewed listings only)",
                         "Entry year implied by the id (all listings, also never reviewed)"],
               loc="upper center", bbox_to_anchor=(0.5, 0.965), ncols=2, frameon=False, fontsize=9)
    fig.suptitle("Share of monthly+ listings (30+ nights) by entry year", y=1.0)
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_monthly_by_year(df, "study06_monthly_by_entry_year.png")

### 6b. Barcelona: tourist licences by cohort

Study 04 found that in Barcelona the licence field separates the two markets: short-stay listings carry a tourist licence (HUTB), monthly+ listings are mostly "exempt: seasonal rental". Barcelona has issued no new tourist licences since 2014 (study 04; the exact date to verify). If the freeze works, listings that entered after 2014 should carry an HUTB licence less often. A licence belongs to the flat, not to the listing, so a new listing can still use an old licence; the effect should therefore be visible but not total. Chi-square test and Cramér's V as in section 6. The licence field has a different format in every city, so this check is for Barcelona only and is not part of the replication.

**Prediction:** the hypothesis comes from the study plan (written before looking at the data): entrants after the 2014 freeze less often have a tourist licence. The shares per entry year were already seen during the first exploration.

**Result:**
- **Tourist licence (HUTB):** **89.2% of the listings that entered before 2015** carry one, against **48.5–58.6% in every later cohort**.
- **Exempt: seasonal rental:** the share grows from 4.7% to 21.7% (2024–2026).
- **Other (hotels, hostels):** 21.0% in the COVID cohort; hotels apparently used Airbnb when tourism collapsed.
- **Test:** Cramér's V = 0.171 (small to medium), chi-square valid (smallest expected count 94).

**Interpretation (draft for the team to check):** the licence freeze leaves a clear fingerprint: the pre-2015 cohort is almost entirely licensed, and after 2014 the licensed share drops at once to about half. That half of new listings still has a licence fits the fact that a licence belongs to the flat: a new listing (new host, new management company, or a re-created listing) can use an existing licence. The rest of the growth goes into seasonal rentals and hotels, the two ways around the freeze.

In [ ]:
# Barcelona only: the licence field has a different format in every city (classification as in study 04)
bcn = df[(df["city"] == "barcelona") & df["cohort"].notna()]
licence = bcn["license"].fillna("").astype(str)
licence_status = pd.Series("other (hotel, hostel, ...)", index=bcn.index, name="licence_status")
licence_status.loc[licence == ""] = "missing"
licence_status.loc[licence.str.contains("HUTB")] = "tourist licence (HUTB)"
licence_status.loc[licence.str.contains("seasonal rental", case=False)] = "exempt: seasonal rental"

licence_table = pd.crosstab(bcn["cohort"], licence_status)
display(pd.crosstab(bcn["cohort"], licence_status, normalize="index").mul(100).round(1).assign(listings=licence_table.sum(axis=1)))
pd.Series(cramers_v(licence_table)).round(3)

### 6c. Los Angeles: why are recent entrants less often monthly+?

Section 6 showed the opposite of the prediction for Los Angeles. Study 04 found that the monthly+ share follows the boundary of the City of Los Angeles (74% inside, 9–22% outside), because the city's Home-Sharing Ordinance applies only inside it. A possible explanation is therefore **where** the new listings enter. The cell shows, per cohort, the share of listings in each of the three areas (City of Los Angeles, other cities of the county, unincorporated areas) and the monthly+ share inside each area. A **shift-share check** then separates the two effects: it computes the monthly+ share the 2024–2026 cohort would have if it had the geographic mix of the 2015–2019 cohort. The difference to the actual share is the part of the fall explained by location; the rest happened inside the areas. This follow-up was added after seeing the result of section 6, so it explains a pattern rather than testing a prediction.

**Result:**
- **Growth moved outside the city boundary.** The share of entrants located in the City of Los Angeles falls from **66.3% (before 2015) to 38.5% (2024–2026)**. Over the same cohorts, the other cities of the county grow from 29.1% to 48.9% and the unincorporated areas from 4.5% to 12.6%.
- **Inside the City of Los Angeles the monthly+ share also falls,** from 76.9–79.9% (cohorts before 2020) to **54.3%** (2024–2026). In the other cities it falls from 38.7% to 11.4%.
- **Shift-share:** with the geographic mix of 2015–2019, the 2024–2026 cohort would be 35.8% monthly+ instead of 27.2%. **Location explains about 29% of the fall** (from 56.7% to 27.2%); the remaining 71% happened inside the areas.

**Interpretation (draft for the team to check):** two mechanisms, both linked to regulation.
1. New supply grows where the Home-Sharing Ordinance does not apply (outside the City of Los Angeles).
2. Inside the city, a survivorship fingerprint: under the ordinance (2019), unregistered short-stay listings had to switch to 30+ or leave. The old listings still online are therefore mostly 30+. New listings enter either as registered short stays or as 30+ from the start.

Part of the within-city fall is also the proxy bias of section 6, since recent monthly+ listings are often unreviewed. The data cannot separate these explanations further.

In [ ]:
# Los Angeles only: the monthly+ share follows the boundary of the City of Los Angeles (study 04)
la = df[(df["city"] == "los_angeles") & df["cohort"].notna()]
mix = pd.crosstab(la["cohort"], la["neighbourhood_group_cleansed"], normalize="index")
monthly_by_area = la.groupby(["cohort", "neighbourhood_group_cleansed"])["is_monthly"].mean().unstack()
display(mix.mul(100).round(1).assign(listings=la.groupby("cohort").size()))
display(monthly_by_area.mul(100).round(1))

# Shift-share check: the monthly+ share of the 2024-2026 cohort if it had the geographic mix of 2015-2019
actual_old = la.loc[la["cohort"] == "2015-2019", "is_monthly"].mean() * 100
actual_new = la.loc[la["cohort"] == "2024-2026", "is_monthly"].mean() * 100
reweighted = (mix.loc["2015-2019"] * monthly_by_area.loc["2024-2026"]).sum() * 100
pd.Series({"monthly_pct_2015_2019": actual_old, "monthly_pct_2024_2026": actual_new,
           "monthly_pct_2024_2026_with_2015_2019_mix": reweighted,
           "share_of_fall_explained_by_location_pct": (reweighted - actual_new) / (actual_old - actual_new) * 100}).round(1)

## 7. Who brings the new listings: new hosts or incumbents?

In a young market, growth comes from new hosts. In a mature market, more of the new listings come from hosts who have been on the platform for years. `host_start_year` comes from the host's time as a host, which counts from the host's first listing anywhere, also listings that no longer exist; so this measure is less affected by survivorship than the entry counts. A listing has an **incumbent host** if the host started hosting at least two years before the listing's entry year. The table shows the incumbent share per cohort. (This is about entry, not about how many listings a host has; host concentration is study 02.)

**Prediction (written before running, by Claude):** in both cities more than half of the listings that entered in 2024–2026 come from incumbent hosts, and the share rises from cohort to cohort.

**Result:** confirmed in both cities. In the 2024–2026 cohort, **73.1% (Barcelona) and 55.7% (Los Angeles)** of the new listings come from hosts who had been hosting for at least two years. The share rises from cohort to cohort: Barcelona 21.3% → 40.0% → 60.2% → 69.9% → 73.1%; Los Angeles 10.7% → 25.5% → 47.6% → 44.1% → 55.7% (one small dip after COVID).

**Interpretation (draft for the team to check):** most new supply in Barcelona comes from hosts who are already on the platform, which is a sign of a mature market. Los Angeles still attracts more new hosts. Two caveats:
1. Part of the rise is mechanical. In 2012 nobody could have hosted for long, because Airbnb itself was only a few years old.
2. "Time as a host" counts from the host's first listing anywhere, so a host from another city counts as an incumbent.

The jump in the COVID cohort fits a market where new hosts stopped entering while existing hosts kept adding (or re-creating) listings.

In [ ]:
def incumbent_share(data: pd.DataFrame) -> pd.DataFrame:
    """Share (%) of listings per cohort whose host had started hosting at least 2 years before the listing's
    entry year (an "incumbent" host), per city."""
    dated = data.dropna(subset=["cohort", "host_start_year"])
    incumbent = dated["host_start_year"] <= dated["entry_year"] - 2
    return incumbent.groupby([dated["cohort"], dated["city_name"]]).mean().mul(100).round(1).unstack("city_name")


incumbent_share(df)

## 8. Where is the market expanding?

### 8a. Distance to the market centre

Is the market spreading outwards? Each listing's distance to its city's **market centre**, defined as the median latitude and median longitude of all of today's listings (a data-driven centre, so no landmark has to be picked by hand). The table gives the median distance per cohort. The **Mann-Whitney U test** compares the distances of new listings (first review in the last 24 months) with established ones: it is the non-parametric test for two groups and does not assume normal distributions (distances are skewed). The effect size is the **rank-biserial r** = 2*U* / (*n*₁*n*₂) − 1: the probability that a random new listing is farther out than a random established one, minus the opposite probability (0 = no difference, > 0 = new listings farther out).

**Prediction (from the study plan, written by Claude before looking at the data):** new listings spread to outer neighbourhoods, so r > 0 in both cities.

**Result:** **not confirmed.**
- **Barcelona:** new listings are a median of 1.64 km from the centre, against 1.55 km for established ones. r = **0.063**: significant (p < 0.001) but negligible. The median distance per cohort stays between 1.43 and 1.66 km.
- **Los Angeles:** 14.37 vs 14.55 km, r = **−0.029**, so new listings are, if anything, slightly closer. Per cohort the median distance first grows (11.9 km before 2015 → 16.5 km in 2022–2023), then drops again (14.5 km in 2024–2026).

With n = 11,745 and 32,494, "significant" here only means "measurable", not "large".

In [ ]:
def distance_by_cohort(data: pd.DataFrame) -> pd.DataFrame:
    """Median distance (km) to the market centre per entry cohort and city."""
    dated = data.dropna(subset=["cohort"])
    return dated.groupby(["cohort", "city_name"])["dist_km"].median().round(2).unstack("city_name")


def spread_test(data: pd.DataFrame) -> pd.DataFrame:
    """Mann-Whitney U test: are new listings (first review in the last 24 months) farther from the market
    centre than established ones? Effect size: rank-biserial r (> 0 = new listings farther out)."""
    rows = {}
    for city, g in data.dropna(subset=["entry_year"]).groupby("city_name"):
        new, old = g.loc[g["is_new"], "dist_km"], g.loc[~g["is_new"], "dist_km"]
        u, p = stats.mannwhitneyu(new, old)
        rows[city] = {"new": len(new), "established": len(old), "median_km_new": new.median(),
                      "median_km_established": old.median(), "U": u, "p_value": p,
                      "rank_biserial_r": 2 * u / (len(new) * len(old)) - 1}
    return pd.DataFrame(rows).T.astype(float).round(3)


display(distance_by_cohort(df))
spread_test(df)

### 8b. Share of new listings per neighbourhood

For each neighbourhood, the share of new listings among its reviewed listings. If the market is spreading, the share should be **higher where there are fewer established listings** (negative Spearman ρ between established listings and the new share) and **higher farther from the centre** (positive ρ with the distance). Neighbourhoods with fewer than 20 reviewed listings are left out. The new share is computed from reviewed listings only; section 2c showed that listings without a review are mostly new, so the true new share is higher, especially in neighbourhoods with many monthly+ listings.

**Prediction (written before running, by Claude):** ρ(established listings, new share) < 0 and ρ(distance, new share) > 0 in both cities.

**Result:**
- **Established listings vs new share, confirmed:** ρ = **−0.298 (Barcelona, p = 0.040)** and **−0.206 (Los Angeles, p = 0.004)**. Neighbourhoods that already have many listings get relatively fewer new ones.
- **Distance vs new share, not confirmed:** ρ = 0.234 in Barcelona (p = 0.11, not significant) and 0.058 in Los Angeles (p = 0.43).
- **Barcelona:** the new share ranges from 24.1% to 52.2%. It is highest in Sarrià (52.2%), Navas, el Coll, el Clot and Sant Gervasi – la Bonanova, and lowest in three Eixample neighbourhoods (la Dreta de l'Eixample 24.5%, el Fort Pienc 24.3%, l'Antiga Esquerra de l'Eixample 24.1%), Hostafrancs and Sant Martí de Provençals.
- **Los Angeles:** the range is wider (10.8–81.7%). The top is Exposition Park (81.7%), Veterans Administration, the Santa Susana Mountains, Universal City and Marina del Rey. Among the lowest are Altadena (18.1%) and Pacific Palisades (17.9%).

**Interpretation (draft for the team to check):** the market does not move **outwards**, but it fills in **sideways**. New listings go relatively more to neighbourhoods that were not yet saturated, while the dense core (the Eixample in Barcelona) grows less in relative terms. One caution on the method: if new listings simply followed the existing ones (proportional growth), ρ would be about 0; a negative ρ means less-than-proportional growth in dense areas. In Los Angeles, Altadena and Pacific Palisades were hit by the January 2025 wildfires, which may explain the few new entries there (to verify). Exposition Park hosts venues of the 2028 Olympic Games, a possible pull factor (to verify).

In [ ]:
MIN_LISTINGS = 20  # neighbourhoods with fewer reviewed listings are left out


def new_share_by_neighbourhood(data: pd.DataFrame) -> pd.DataFrame:
    """Per neighbourhood: reviewed listings, new and established listings, share of new listings (%) and
    the median distance to the market centre."""
    dated = data.dropna(subset=["entry_year"])
    t = (dated.groupby(["city_name", "neighbourhood_cleansed"])
         .agg(district=("neighbourhood_group_cleansed", "first"), listings=("id", "size"),
              new=("is_new", "sum"), dist_km=("dist_km", "median"))
         .reset_index())
    t["established"] = t["listings"] - t["new"]
    t["new_share"] = t["new"] / t["listings"] * 100
    return t[t["listings"] >= MIN_LISTINGS].reset_index(drop=True)


def spread_correlation(t: pd.DataFrame) -> pd.DataFrame:
    """Spearman correlations per city between the share of new listings and (a) the number of established
    listings, (b) the distance to the market centre."""
    rows = {}
    for city, g in t.groupby("city_name"):
        rho_e, p_e = stats.spearmanr(g["established"], g["new_share"])
        rho_d, p_d = stats.spearmanr(g["dist_km"], g["new_share"])
        rows[city] = {"neighbourhoods": len(g), "new_share_min": g["new_share"].min(),
                      "new_share_max": g["new_share"].max(), "rho_established_vs_new_share": rho_e,
                      "p_established": p_e, "rho_distance_vs_new_share": rho_d, "p_distance": p_d}
    return pd.DataFrame(rows).T.astype(float).round(3)


def top_new(t: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """The n neighbourhoods with the highest and the n with the lowest share of new listings, per city."""
    ranked = t.sort_values("new_share", ascending=False)
    top = ranked.groupby("city_name").head(n).assign(group="highest")
    bottom = ranked.groupby("city_name").tail(n).assign(group="lowest")
    columns = ["city_name", "group", "neighbourhood_cleansed", "district", "listings", "new_share", "dist_km"]
    return pd.concat([top, bottom])[columns].sort_values(["city_name", "group", "new_share"],
                                                          ascending=[True, True, False]).round(1)


nbhd = new_share_by_neighbourhood(df)
display(spread_correlation(nbhd))
top_new(nbhd)

### 8c. Static maps per cohort (report figure)

One row per city, one panel per cohort: grey dots are all of today's reviewed listings, coloured dots the listings of the cohort, thin lines the neighbourhood borders. This is the static version of the animated map in section 9, which can only show one frame in a screenshot. Each panel shows at most 1,000 random listings of the cohort, so a large cohort does not look denser just because it is larger; the panel title gives the real size.

**Result:**
- **Barcelona:** all five cohorts sit in the same area (Eixample, Ciutat Vella, Gràcia, Poblenou). The newest cohort reaches somewhat more into the northern districts, but the overall pattern hardly moves.
- **Los Angeles:** the listings that entered before 2015 cluster on the coastal Westside (Venice, Santa Monica) and in Hollywood. Later cohorts spread in all directions: the San Fernando Valley, the San Gabriel Valley in the east, the South Bay and, in 2022–2026, the north of the county (Santa Clarita, Antelope Valley). Because the spread goes in all directions around a polycentric city, the median distance to one centre (section 8a) hardly changes.

In [ ]:
NAME_TO_KEY = {c.name: k for k, c in CITIES.items()}


def polygon_outlines(city: str) -> list:
    """Outer rings of the neighbourhood polygons of `city`, as (longitude, latitude) arrays for plotting."""
    lines = []
    for feature in load_neighbourhoods(city)["features"]:
        geom = feature["geometry"]
        if geom is None:
            continue
        polygons = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
        for polygon in polygons:
            ring = np.asarray(polygon[0])
            lines.append((ring[:, 0], ring[:, 1]))
    return lines


def plot_cohort_maps(data: pd.DataFrame, filename: str, sample: int = 1000) -> None:
    """Static small multiples: one row per city, one panel per entry cohort. Grey dots are all of today's
    reviewed listings, coloured dots a random sample of `sample` listings of the cohort (all of them if the
    cohort is smaller), so that panels show where a cohort is, not how big it is. Lines: neighbourhood borders."""
    dated = data.dropna(subset=["cohort"])
    city_names = dated["city_name"].unique()
    fig, axes = plt.subplots(len(city_names), len(COHORT_LABELS), figsize=(16, 3.5 * len(city_names)), squeeze=False)
    for row, city in zip(axes, city_names):
        g = dated[dated["city_name"] == city]
        lat, lon = g["latitude"].quantile([0.01, 0.99]), g["longitude"].quantile([0.01, 0.99])
        pad_lat, pad_lon = (lat.iloc[1] - lat.iloc[0]) * 0.05, (lon.iloc[1] - lon.iloc[0]) * 0.05
        outlines = polygon_outlines(NAME_TO_KEY[city])
        for ax, cohort in zip(row, COHORT_LABELS):
            for x, y in outlines:
                ax.plot(x, y, color="#c3c2b7", linewidth=0.3)
            ax.scatter(g["longitude"], g["latitude"], s=0.4, color="#bdbdbd", alpha=0.5, linewidth=0, rasterized=True)
            c = g[g["cohort"] == cohort]
            shown = c.sample(min(sample, len(c)), random_state=0)
            ax.scatter(shown["longitude"], shown["latitude"], s=2, color=COHORT_COLORS[cohort], linewidth=0,
                       rasterized=True)
            ax.set_xlim(lon.iloc[0] - pad_lon, lon.iloc[1] + pad_lon)
            ax.set_ylim(lat.iloc[0] - pad_lat, lat.iloc[1] + pad_lat)
            ax.set_aspect(1 / np.cos(np.radians(g["latitude"].median())))  # equal distances in both directions
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_title(f"{cohort} ({len(c) / len(g):.0%}, n = {len(c):,})", fontsize=9)
        row[0].set_ylabel(city, fontsize=12)
    fig.suptitle(f"Where did today's listings enter? Up to {sample:,} random listings per entry cohort "
                 "(grey: all reviewed listings)", y=1.0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_cohort_maps(df, "study06_cohort_maps.png")

## 9. Maps

Two interactive maps per city, saved in `maps/` as HTML:
1. `study06_new_listings_<city>.html`:
   - **choropleth** of the share of new listings (first review in the last 24 months) per neighbourhood, one blue hue from light (low) to dark (high); neighbourhoods with fewer than 20 reviewed listings are grey;
   - **categorical point map**: one layer per entry cohort (orange, light = old, dark = new), with **marker clustering**: nearby points are merged into a circle with their count, which splits up when zooming in. This keeps a map with tens of thousands of listings fast. Hovering a point shows the month of the first review. The cohort layers are off when the map opens and can be switched on one by one, e.g. "before 2015" and then "2024-2026";
   - **heat map** of the new listings (off when the map opens).
2. `study06_entry_animation_<city>.html`: an **animated heat map** with one frame per year of the first review (2012–2026); the slider and the play button sit in the bottom-left corner. Each frame shows only the listings that entered in that year.

Interactive features: three base maps (TMS), layer control, custom colour scales, tooltips on hover and popups on click (choropleth). The new helpers `add_category_points` and `add_heatmap_over_time` live in `src/maps.py` and work for every city. Screenshots for the report: `figures/study06_map_new_share_<city>.png`.

**Result:**
- **Barcelona:** the lightest neighbourhoods (lowest share of new listings, about 24%) are the Eixample and its surroundings. The darkest (about 50%) form a broken ring around them: the upper districts (Sarrià, Sant Gervasi – la Bonanova, el Coll), el Clot and Navas in the east, and the edge of Sants-Montjuïc in the south. This is the "filling in sideways" of section 8b.
- **Los Angeles:** high shares appear both on the periphery (the north of the county, the Santa Susana Mountains, the east of the San Gabriel Valley) and in a few central spots (Exposition Park, Universal City). There is no single centre-to-edge gradient, which matches the near-zero correlation with distance.
- The first frame of the animated map (2012) shows the few surviving early listings in the centre. The later frames are best compared in the static panels of section 8c.

In [ ]:
def make_maps(data: pd.DataFrame, t: pd.DataFrame, show: bool = True) -> None:
    """Two interactive maps per city (see the description above)."""
    for city in data["city"].unique():
        city_df = data[data["city"] == city]
        dated = city_df.dropna(subset=["cohort"])
        city_t = t[t["city_name"] == CITIES[city].name]

        # Map 1: share of new listings per neighbourhood, cohort points and a heat map of new listings
        m = base_map(city_df)
        add_choropleth(m, city, city_t.set_index("neighbourhood_cleansed")["new_share"],
                       dated.groupby("neighbourhood_cleansed").size(), name="Share of new listings",
                       legend=f"Share of new listings (%, first review in the last {NEW_MONTHS} months)",
                       min_count=MIN_LISTINGS)
        add_category_points(m, dated.assign(label="First review " + dated["first_review"].dt.strftime("%Y-%m")),
                            category="cohort", colors=COHORT_COLORS, label="label", name_prefix="Entered ")
        add_heatmap(m, dated[dated["is_new"]], name="Heat map: new listings")
        path = save_map(m, f"study06_new_listings_{city}.html")
        print("Saved", path.name)
        if show:
            display(m)

        # Map 2: animated heat map, one frame per entry year
        m2 = base_map(city_df)
        add_heatmap_over_time(m2, dated[dated["entry_year"] >= 2012], period="entry_year",
                              name="Listings by year of the first review")
        path = save_map(m2, f"study06_entry_animation_{city}.html")
        print("Saved", path.name)


make_maps(df, nbhd)

# Replication: pairs 2 and 3

The method is frozen after pair 1: the cells below load all six cities and call **the same functions** with the same definitions (cohort bins, 24 months for "new", 2019 as the base year, fit years 2010–2025, 20 listings per neighbourhood, exit rates of the reference band). Pair 2 (Madrid – New York) and pair 3 (Tokyo – Mexico City) test whether the pair-1 findings hold elsewhere.

**Predictions before running the replication:** the yearly entry counts of all six cities were seen during the first exploration, so the entry-curve predictions below are **not blind**. They are the study plan's predictions (written before looking at the data):
- a COVID dip in 2020 in every city;
- in New York, mostly monthly+ entrants after Local Law 18 (in force since September 2023).

Written before running, by Claude (not seen before): new listings are farther from the centre than established ones in every city (r > 0); more than half of the 2024–2026 entrants come from incumbent hosts in every city.

## R1. Load all six cities

In [ ]:
df_all = add_entry_columns(load_cities(list(CITIES)))
counts_all = entries_per_year(df_all)
nbhd_all = new_share_by_neighbourhood(df_all)
pd.DataFrame({"listings": df_all.groupby("city_name").size(),
              "reviewed_listings": df_all.dropna(subset=["entry_year"]).groupby("city_name").size(),
              "neighbourhoods_with_20+": nbhd_all.groupby("city_name").size()})

## R2. Key indicators per city

The main numbers of sections 2–8 for all six cities, ordered by pair.

In [ ]:
tests_all = cohort_tests(df_all)
incumbents_all = incumbent_share(df_all)
growth_all = growth_models(counts_all)
summary = pd.concat([
    unreviewed_profile(df_all)[["unreviewed_pct", "unreviewed_implied_2025_26_pct"]],
    covid_dip(counts_all)[["change_2020_vs_2019_pct", "change_2021_vs_2019_pct", "peak_year_to_2025"]],
    momentum(counts_all)[["ratio_2025_vs_2023", "reading"]],
    growth_all[["best_by_aic", "inflection_year_t0", "t0_se", "K_vs_count_2025"]],
    tests_all.xs("cohort x monthly+", level=1)[["cramers_v", "valid"]].add_suffix("_monthly"),
    tests_all.xs("cohort x entire home", level=1)[["cramers_v", "valid"]].add_suffix("_entire_home"),
    incumbents_all.loc["2024-2026"].rename("incumbent_pct_2024_26"),
    spread_test(df_all)[["rank_biserial_r"]],
    spread_correlation(nbhd_all)[["rho_established_vs_new_share", "rho_distance_vs_new_share"]],
], axis=1).reindex([c.name for c in CITIES.values()])
summary.insert(0, "pair", [c.pair for c in CITIES.values()])
summary

## R3. Tables and charts for all six cities

Entry cohorts and the COVID dip (section 3), the index with the survivorship band and the momentum check (section 4), the growth models (section 5), and the cohort profile, chi-square tests, incumbent shares and monthly+ share by entry year (sections 6–7), drawn by the same functions.

In [ ]:
display(cohort_table(df_all))
display(covid_dip(counts_all))
plot_entries(counts_all, "study06_entries_per_year_all.png")

In [ ]:
display(entry_index(counts_all).loc[[2015, 2017, 2019, 2020, 2021, 2023, 2025]].round(0).T)
display(momentum(counts_all))
plot_index(counts_all, "study06_entries_index_all.png")

In [ ]:
display(growth_all)
plot_growth_fits(counts_all, "study06_growth_fits_all.png")

In [ ]:
display(cohort_profile(df_all))
display(tests_all)
display(incumbents_all)
plot_monthly_by_year(df_all, "study06_monthly_by_entry_year_all.png")

## R4. Geography and maps

Distance to the market centre (section 8a), the share of new listings per neighbourhood (8b), the static cohort maps (8c) and the interactive maps (section 9), saved in `maps/`.

In [ ]:
display(distance_by_cohort(df_all))
display(spread_test(df_all))
display(spread_correlation(nbhd_all))
top_new(nbhd_all)

In [ ]:
plot_cohort_maps(df_all, "study06_cohort_maps_all.png")

In [ ]:
make_maps(df_all, nbhd_all, show=False)

### R4b. New York: who are the recent entrants below 30 nights?

The prediction for New York was mostly monthly+ entrants after Local Law 18 (in force since September 2023). Under the law, a stay below 30 days is legal only in a registered unit with the host present, and the registration number (`OSE-STRREG-...`) appears in the `license` field; Airbnb also lets a listing declare an exemption (for example a hotel; the exact exemption categories are to verify). The cell takes the listings that are **not** monthly+ and shows, per cohort, their licence status, their share of the cohort and the share of hotel rooms among them. This is a city-specific follow-up (the licence field has a different format in every city), added after seeing the result in R3.

**Result:**
- **Fewer 30+ entrants than predicted.** Listings below 30 nights are **9% of the cohorts before 2020**, 21% of 2020–2023 and **41.8% of the 2024–2026 cohort** (2,128 listings). The prediction "mostly monthly+ after 2023" holds only narrowly: 58.2% of the 2024–2026 cohort is monthly+ by first review (66–79% per year by the id-implied year), the lowest share of any cohort.
- **The new short-stay entrants are mostly registered.** Of the 2024–2026 listings below 30 nights, **40.7% carry an OSE-STRREG registration number and 47.5% declare an exemption**; only 11.8% have no licence entry. Hotel rooms are only 5.3% of them, so the exemptions are not mainly hotels (what they are is to verify).
- **The old cohorts are almost entirely 30+.** Among the few old listings below 30 nights, 67–70% are registered.

**Interpretation (draft for the team to check):** this looks like Local Law 18 seen through survivorship.
- **The old cohorts.** When enforcement started in 2023, unregistered short-stay listings had to switch to 30+ or disappear; Airbnb reportedly removed thousands of them (to verify). The listings that survived from before 2023 are therefore 30+ (91%).
- **The new cohort.** New short-stay listings did not stop entering. They entered **inside** the registration system. The law did not end short stays; it filtered them.

The same pattern appears in the City of Los Angeles (section 6c), which has a registration system too. In both cases the old cohorts are 30+ because the short-stay listings without registration are gone, not because 30+ was popular in 2015.

In [ ]:
# New York only: registration status of the listings that are NOT monthly+, per cohort
ny = df_all[(df_all["city"] == "new_york") & df_all["cohort"].notna()]
ny_short = ny[~ny["is_monthly"]]
ny_licence = ny_short["license"].fillna("").astype(str)
ny_status = pd.Series("other", index=ny_short.index, name="licence_status")
ny_status.loc[ny_licence == ""] = "missing"
ny_status.loc[ny_licence.str.lower().str.startswith("ose-strreg")] = "STR registration (OSE-STRREG)"
ny_status.loc[ny_licence.str.contains("exempt", case=False)] = "exempt"
pd.crosstab(ny_short["cohort"], ny_status, normalize="index").mul(100).round(1).assign(
    below_30_nights=ny_short.groupby("cohort").size(),
    share_of_cohort_pct=(ny_short.groupby("cohort").size() / ny.groupby("cohort").size() * 100).round(1),
    hotel_room_pct=ny_short.groupby("cohort")["room_type"].agg(lambda s: (s == "Hotel room").mean() * 100).round(1))

## R5. Replication result

Each pair-1 finding checked on pairs 2 and 3 with the same code. **Replicated** = holds in both cities of the pair, **Partial** = holds in one of them, **Not replicated** = holds in neither.

| Pair-1 finding (Barcelona, Los Angeles) | Pair 2: Madrid – New York | Pair 3: Tokyo – Mexico City |
|---|---|---|
| Listings without a review are mostly new: 57%, 60% have an id-implied entry year of 2025–2026 | **Partial:** Madrid 50.4%, New York 45.1% | **Replicated:** 84.0%, 76.5% |
| COVID dip: entries fell from 2019 to 2020 (−49%, −30%) | **Replicated:** −61%, −43% | **Replicated:** −40%, −39% |
| Recovery: entries back above 2019 already in 2021 (+3%, +13%) | **Partial:** New York +21%; Madrid still −13% | **Partial:** Mexico City +1%; Tokyo still −43% |
| Most of today's stock is young: 37%, 45% entered in 2024–2026 | **Partial:** Madrid 49.8%; New York only 23.4% | **Replicated:** Tokyo 69.3%, Mexico City 48.8% |
| Linear growth is rejected; logistic and exponential fit well (R² ≥ 0.985) | **Replicated:** linear R² 0.80, 0.94; logistic 0.996, 0.994 | **Replicated:** linear R² 0.63, 0.78; logistic 0.992, 0.998 |
| The stricter city of the pair shows a slowdown, the less strict one does not (Barcelona: inflection 2024.6 ± 1.5, momentum 1.34 = compatible with a flat market; Los Angeles: no inflection, momentum 1.67 = growing) | **Replicated:** New York (stricter) has the clearest slowdown of all six cities: inflection 2022.3 ± 0.7, momentum 0.74 = shrinking; Madrid: no inflection (exponential best), momentum 1.55 | **Not replicated (reversed):** Tokyo (stricter) grows fastest of all six (momentum 2.95, no inflection); Mexico City (less strict) bends (inflection 2026.0 ± 0.9, momentum 1.42) |
| The monthly+ share by cohort differs between the cities of the pair: rises in the European city (8% → 38%), falls in the US city (63% → 27%) (V = 0.22, 0.25) | **Replicated (same split):** Madrid rises 4–6% → 9% (V = 0.08, negligible); New York falls 91% → 58% (V = 0.30, the largest effect) | **Partial:** Tokyo rises 0–2% → 9% (V = 0.14); Mexico City has almost no monthly+ listings (test invalid by Cochran's rule) |
| Room type barely differs by cohort (V = 0.13, 0.09) | **Replicated:** V = 0.05, 0.06 | **Replicated:** V = 0.14, 0.04 |
| Most 2024–2026 entrants come from incumbent hosts (73%, 56%), rising by cohort | **Replicated:** 55.4%, 60.3% | **Not replicated:** Tokyo 41.1%, Mexico City 49.6% (still rising by cohort) |
| New listings are not farther from the centre (r = 0.06, −0.03) | **Partial:** New York r = 0.00; Madrid r = 0.13 (small, the only city that spreads outwards: median 1.05 km before 2015 → 2.21 km in 2024–2026) | **Replicated:** r = 0.05, −0.01 |
| New listings go relatively more to less saturated neighbourhoods (ρ = −0.30, −0.21) | **Partial:** Madrid −0.30 (p = 0.002); New York −0.01 (n.s.) | **Not replicated:** +0.24, +0.20 (both n.s.; only 33 and 16 units) |
| Study-plan prediction for New York: mostly monthly+ entrants after Local Law 18 | – | Narrowly: 58% of the 2024–2026 cohort, the lowest of any cohort; the rest is mostly registered or exempt (R4b) |

**Interpretation (draft for the team to check):**
- **What all six cities share:** a COVID dip in 2020 (−30% to −61%), a young stock (except New York), and survivor curves that no linear model can describe. Room type hardly changes between cohorts. These look like properties of the platform and of the snapshot method.
- **The growth verdict depends on regulation in pairs 1 and 2, but not in pair 3.**
  - In pair 1 the stricter Barcelona is flatter than Los Angeles. In pair 2, where the direction of regulation is reversed, the stricter New York is the flattest city of all (the only one with fewer entries in 2025 than in 2023), while Madrid keeps growing.
  - So "Europe vs USA" does not explain the pattern, and regulation does, as in study 04.
  - Pair 3 breaks the rule: Tokyo, the stricter city, grows fastest. Two candidate explanations:
    1. the 2018 registration law (minpaku) reset Tokyo's market, so almost all of today's stock is post-2018 (only 0.2% before 2015);
    2. tourism to Japan boomed after the borders reopened in October 2022.

    Both are to verify. The 2021 figure (−43% vs 2019, the only city without a recovery) fits the long border closure.
- **Regulation leaves a survivorship fingerprint.** In New York and in the City of Los Angeles, the old cohorts are mostly 30+ and the new short-stay entrants mostly registered. The rules did not freeze the market; they replaced the unregistered short-stay listings with registered ones and 30+ ones. In Barcelona, the licence freeze makes the old cohorts the licensed ones.
- **Geography:** the market fills in sideways rather than moving outwards. Madrid is the only city with a visible outward spread.

**Open questions for the team:**
- Can an older Inside Airbnb snapshot (e.g. 2023) measure the real exit rate, and so replace the assumed 10–20% band?
- Is Madrid's outward spread a response to Madrid's 2025 limits on tourist flats in the centre (to verify), or simply the growth of a younger market?
- Do the New York "exempt" listings follow a legal category (e.g. hotels or a specific building type), or is it a self-declaration that nobody checks?

## Findings

### Results on pair 1 (discovery: Barcelona, Los Angeles)
- **Data:** `host_since` is empty in the 2026 data. Host tenure comes from `hosts_time_as_host_*`, and the listing's entry date from `first_review` (missing for 23–26%).
  - The listing id works as a clock: ρ = 0.96 with the first-review date, and within one year for 92–95% of listings.
  - With it, **57–60% of the unreviewed listings turn out to be from 2025–2026**, and most of them are monthly+. So the first-review proxy under-counts the newest and the monthly+ listings.
- **Entry timing:** 37% (Barcelona) and 45% (Los Angeles) of today's listings entered in 2024–2026. Entries fell by 49% and 30% in 2020 (COVID), recovered in 2021, and have risen every year since.
- **Growth or churn?**
  - **Index:** 2025 = 216 (Barcelona) and 362 (Los Angeles), with 2019 = 100. Both lie inside the band a flat market with 10–20% yearly exits would produce.
  - **Momentum** (2025 vs 2023): 1.34 in Barcelona, compatible with a flat market; 1.67 in Los Angeles, growth.
  - **Logistic fit:** Barcelona bends around now (inflection 2024.6 ± 1.5); Los Angeles shows no slowdown (no inflection before 2028). A linear model is rejected in both cities.
- **Who enters:**
  - **Monthly+ share by cohort:** Barcelona rises from 8% to 38%, while the share of tourist licences falls from 89% (before 2015) to about 50%; this is the 2014 licence freeze. Los Angeles falls from 63% to 27%; growth moved outside the City of Los Angeles (location explains 29% of the fall) and fell within it.
  - **Room type** barely changes.
  - **Hosts:** 73% (Barcelona) and 56% (Los Angeles) of the 2024–2026 entrants come from incumbent hosts.
- **Where:** new listings are not farther from the centre (r = 0.06 and −0.03, negligible). They go relatively more to neighbourhoods with fewer established listings (ρ = −0.30 and −0.21); in Barcelona, that means outside the Eixample.

### Cross-city comparison (replication on pairs 2 and 3)
- **Replicated everywhere:** a COVID dip (−30% to −61%), non-linear survivor curves, little change in room type between cohorts.
- **Growth verdict:**
  - New York is the only shrinking market (entries 2025/2023 = 0.74, inflection 2022.3 ± 0.7), consistent with Local Law 18.
  - Tokyo grows fastest (2.95), after its 2018 registration reset and the 2022 reopening (to verify).
  - Barcelona, Madrid and Mexico City are compatible with a flat market under the survivorship band; Los Angeles grows modestly.
- **Regulation, not continent:** within pairs 1 and 2 the stricter city is the flatter one, although the direction of regulation is reversed in pair 2. Pair 3 does not follow this rule.
- **Survivorship fingerprint of registration systems:** in New York and in the City of Los Angeles the old cohorts are mostly 30+, and the new short-stay entrants are mostly registered or exempt (New York 2024–2026: 88%).
- **Main conclusion (draft):** a single snapshot can say **when today's listings entered**, but it can tell growth from churn only where the signal is strong. It shows a shrinking market in New York and a booming one in Tokyo. For the other four cities the answer depends on the unknown exit rate. The clearest results are about **composition**: who enters (licensed vs seasonal, registered vs 30+, incumbents vs new hosts) follows each city's rules.

### Limitations
- **Survivorship:** listings that left before June 2026 are invisible, so every curve shows survivors, and older years are under-counted. The exit rates of the reference band (10–20% per year) are assumptions.
- **Entry proxy:** `first_review` comes weeks or months after a listing is created (the COVID years are especially affected), and it is missing for unreviewed listings, which are mostly new and monthly+. The id-implied year reduces but does not remove this bias.
- **Today's values:** `minimum_nights`, `room_type` and `license` are values at the scrape, not at entry. A cohort's profile therefore mixes what entered with what survived and changed.
- **Model statistics:** the logistic fit is on cumulative counts, whose errors are correlated from year to year. The standard errors of *t*₀ and *K* are therefore too optimistic, and *K* is not a forecast.
- **Host tenure** counts from the host's first listing anywhere (other cities, deleted listings). The month-to-date conversion is approximate.
- **Geography:** the "market centre" is a single point, which is crude for polycentric Los Angeles and Tokyo. Neighbourhood sizes differ (16 boroughs in Mexico City, 190 neighbourhoods with 20+ listings in Los Angeles).
- **Correlation, not causation:** regulation, tourism shocks (COVID, Japan's border closure), wildfires and the Olympics all coincide in time. With one city per role, the pair design suggests explanations, but it cannot prove them.